# BOARDROOM — GRPO Training
**Runtime → Change runtime type → T4 GPU** before running.

Cells in order:
1. Check GPU
2. Install deps
3. Clone repo + mount Drive
4. W&B login
5. Run training

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}  |  GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Install deps ───────────────────────────────────────────
# Unsloth first (has specific torch version requirements)
!pip install -q unsloth

# Training stack
!pip install -q \
    "trl>=0.12.0" \
    "peft>=0.13.0" \
    "accelerate>=0.34.0" \
    "datasets>=3.0.0" \
    "bitsandbytes>=0.44.0" \
    "wandb>=0.18.0" \
    "matplotlib>=3.8.0"

# OpenEnv (needed for the environment base classes)
!pip install -q "openenv-core[core]>=0.2.2"

print('Done. Restart runtime if prompted.')

In [ ]:
# ── Cell 3: Clone repo + mount Google Drive ────────────────────────
import os

GITHUB_REPO = "https://github.com/YOUR_USERNAME/boardroom.git"  # ← change this
REPO_DIR    = "/content/boardroom"

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# Mount Drive for persistent checkpoint saving
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = "/content/drive/MyDrive/boardroom-checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints → {CHECKPOINT_DIR}')

In [ ]:
# ── Cell 4: W&B login ─────────────────────────────────────────────
import wandb
# Paste your key from wandb.ai/settings when prompted
wandb.login()
print('W&B ready')

In [ ]:
# ── Cell 5: Run training ──────────────────────────────────────────
import sys, argparse
sys.path.insert(0, REPO_DIR)

import train.train_grpo as tg

# ── Config (edit here) ────────────────────────────────────────────
MODEL_ID         = "Qwen/Qwen3.5-4B"   # or Qwen3.5-0.8B for a faster test run
MAX_STEPS        = 200
BATCH_SIZE       = 4
NUM_GENERATIONS  = 8
REFRESH_EVERY    = 50
N_EPISODES       = 50
RUN_NAME         = "boardroom-grpo-v1"
# ─────────────────────────────────────────────────────────────────

args = argparse.Namespace(
    model_id        = MODEL_ID,
    output_dir      = CHECKPOINT_DIR,
    run_name        = RUN_NAME,
    max_steps       = MAX_STEPS,
    batch_size      = BATCH_SIZE,
    num_generations = NUM_GENERATIONS,
    refresh_every   = REFRESH_EVERY,
    n_rollout_episodes = N_EPISODES,
)

# Patch parse_args so main() uses our args instead of sys.argv
tg.parse_args = lambda: args
tg.main()

In [ ]:
# ── Cell 6: Show plots inline ─────────────────────────────────────
from IPython.display import Image, display
from pathlib import Path

for png in ['loss.png', 'reward.png']:
    p = Path(CHECKPOINT_DIR) / png
    if p.exists():
        print(png)
        display(Image(str(p)))